# Qwen local, por bloques

El mismo recorrido del cuaderno de referencia: **configuración → modelo → prompt → cadena → pregunta**.
Al final se inicia la conexión para usar el asistente desde el frontend de Mundus.

Selecciona el kernel `.venv-asistente/Scripts/python.exe` y ejecuta las celdas en orden.
Puedes usar el entorno ya preparado en `Git Tavi/GPT/.venv-asistente`.
Ollama debe estar abierto con `qwen2.5:7b` instalado.

## 1. Configuración

Estas variables eligen el modelo y los puertos. `MAX_SALIDA` corresponde a la pregunta de prueba;
el preinforme del formulario utiliza los límites y la validación de `Asistente/app.py`.
El puerto del puente debe coincidir con `ASISTENTE_PUERTO` del `.env` usado por Docker.

In [ ]:
import os
import sys
from pathlib import Path
from time import perf_counter

os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGSMITH_TRACING"] = "false"

import httpx
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

RAIZ = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "Asistente/app.py").is_file() and (p / "Backend").is_dir()),
    None,
)
if RAIZ is None:
    raise RuntimeError("Abre el cuaderno desde la carpeta entrega 1.")

load_dotenv(RAIZ / "Asistente/.env", override=False)
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://127.0.0.1:11434").rstrip("/")
MODELO = os.getenv("LLM_LOCAL", "qwen2.5:7b")
CONTEXTO = int(os.getenv("LLM_CONTEXTO", "8192"))
MAX_SALIDA = 512
TIMEOUT = float(os.getenv("ASISTENTE_TIMEOUT_SEGUNDOS", "180"))
PUERTO = int(os.getenv("ASISTENTE_PUERTO", "8002"))
print("Proyecto:", RAIZ)
print("Modelo:", MODELO)
print("Puente del frontend:", f"http://127.0.0.1:{PUERTO}")

## 2. Comprobar Ollama

Esta celda revisa el modelo instalado. Si falla, abre Ollama y vuelve a ejecutarla.

In [ ]:
with httpx.Client(timeout=5, trust_env=False) as cliente:
    comprobacion = cliente.get(f"{OLLAMA_URL}/api/tags")
    comprobacion.raise_for_status()

disponibles = {m["name"] for m in comprobacion.json().get("models", [])}
if MODELO not in disponibles:
    raise RuntimeError(f"{MODELO} no está instalado en Ollama.")
print("Ollama y Qwen están disponibles.")

## 3. Crear el modelo

`ChatOllama` conecta LangChain con el Qwen que ejecuta Ollama.

In [ ]:
llm = ChatOllama(
    model=MODELO,
    base_url=OLLAMA_URL,
    temperature=0,
    num_ctx=CONTEXTO,
    num_predict=MAX_SALIDA,
    keep_alive="5m",
    client_kwargs={"timeout": TIMEOUT, "trust_env": False},
)

## 4. Preparar el prompt

Este prompt sirve para la pregunta libre de prueba. El formulario utiliza el prompt documental
de `app.py`, que recibe fragmentos de las guías y exige referencias válidas.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente docente en español. Responde de forma breve y clara. "
               "Si falta información, indícalo."),
    ("human", "{input}"),
])

## 5. Construir la cadena

La salida del prompt pasa al modelo, igual que en el ejemplo de clase.

In [ ]:
conversation = prompt | llm

## 6. Hacer una pregunta

**Cambia `pregunta` y vuelve a ejecutar esta celda.** La respuesta aparece debajo.
Esta prueba libre no recupera guías; el formulario del apartado siguiente sí lo hace.

In [ ]:
pregunta = "¿Qué diferencia hay entre ejecutar una LLM y entrenarla? Responde en dos frases."

inicio = perf_counter()
respuesta = conversation.invoke({"input": pregunta})
print(respuesta.content)
uso = respuesta.usage_metadata or {}
print("\nTokens de entrada:", uso.get("input_tokens", "no reportados"))
print("Tokens de salida:", uso.get("output_tokens", "no reportados"))
print(f"Tiempo: {perf_counter() - inicio:.2f} segundos")

## 7. Conectar el cuaderno con el frontend

Esta celda inicia el puente HTTP de `app.py` con el mismo Python y modelo del cuaderno.
Se ejecuta en un proceso separado para que Jupyter siga disponible.

**Frontend → Java busca fragmentos → puente Python → Qwen → preinforme en pantalla.**

Con Docker iniciado, abre http://localhost:8082/app/. Primero procesa las guías si la biblioteca
está vacía y después utiliza **Consulta de proyecto**. Mantén abierto el kernel.
El puente escucha en `0.0.0.0` para que el backend de Docker pueda acceder al equipo.

In [ ]:
import socket
import subprocess
import time

def detener_asistente():
    proceso = globals().get("proceso_api")
    if proceso is not None and proceso.poll() is None:
        if os.name == "nt":
            subprocess.run(
                ["taskkill", "/PID", str(proceso.pid), "/T", "/F"],
                check=True, capture_output=True,
                creationflags=subprocess.CREATE_NO_WINDOW,
            )
        else:
            proceso.terminate()
        proceso.wait(timeout=10)
    print("Puente detenido.")

def iniciar_asistente():
    global proceso_api
    proceso = globals().get("proceso_api")
    if proceso is not None and proceso.poll() is None:
        print(f"El puente ya está iniciado en el puerto {PUERTO}.")
        return
    if not 1024 <= PUERTO <= 65535:
        raise ValueError("Utiliza un puerto entre 1024 y 65535.")
    with socket.socket() as conexion:
        if conexion.connect_ex(("127.0.0.1", PUERTO)) == 0:
            raise RuntimeError(f"El puerto {PUERTO} está ocupado. No se detuvo ninguna otra aplicación.")

    entorno = os.environ.copy()
    entorno.update({
        "OLLAMA_URL": OLLAMA_URL,
        "LLM_LOCAL": MODELO,
        "LLM_CONTEXTO": str(CONTEXTO),
        "ASISTENTE_TIMEOUT_SEGUNDOS": str(TIMEOUT),
        "PYTHONIOENCODING": "utf-8",
        "PYTHONDONTWRITEBYTECODE": "1",
    })
    log = RAIZ / "output/asistente/jupyter.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    with log.open("a", encoding="utf-8") as archivo:
        proceso_api = subprocess.Popen(
            [sys.executable, "-m", "uvicorn", "Asistente.app:app",
             "--host", "0.0.0.0", "--port", str(PUERTO)],
            cwd=RAIZ, env=entorno, stdin=subprocess.DEVNULL,
            stdout=archivo, stderr=subprocess.STDOUT,
            creationflags=subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0,
        )
    try:
        with httpx.Client(timeout=5, trust_env=False) as cliente:
            for _ in range(60):
                if proceso_api.poll() is not None:
                    raise RuntimeError(f"El puente no arrancó. Revisa {log}.")
                try:
                    salud = cliente.get(f"http://127.0.0.1:{PUERTO}/salud")
                except httpx.TransportError:
                    time.sleep(0.25)
                    continue
                salud.raise_for_status()
                print(salud.json())
                print("Abre http://localhost:8082/app/ y utiliza Consulta de proyecto.")
                return
        raise RuntimeError(f"El puente no respondió. Revisa {log}.")
    except Exception:
        detener_asistente()
        raise

iniciar_asistente()

## 8. Detener el puente al terminar

Descomenta y ejecuta la siguiente línea **antes de cerrar o reiniciar el kernel**.
Al cambiar modelo, contexto o servidor Ollama, detén el puente y vuelve a ejecutar
las celdas de configuración y conexión. La pregunta libre puede repetirse sin reiniciarlo.

In [ ]:
# detener_asistente()